In [0]:
# Force Delta Lake to accept special characters like (), % and spaces
spark.conf.set("spark.databricks.delta.properties.defaults.columnMapping.mode", "name")

from pyspark.sql.functions import col, current_timestamp, current_date

dbutils.widgets.text("catalog", "dbr_dev")
dbutils.widgets.text("login", "valeriimatviiv")
dbutils.widgets.text("dataset_name", "spotify_data")

# Alias variables
CONFIG = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "login": dbutils.widgets.get("login").strip(),
    "dataset": dbutils.widgets.get("dataset_name").strip()
}

bronze_schema = f"{CONFIG['login']}_bronze"
target_table = f"{CONFIG['catalog']}.{bronze_schema}.{CONFIG['dataset']}_table_bronze"

PATHS = {
    "source_volume": f"/Volumes/{CONFIG['catalog']}/{bronze_schema}/{CONFIG['dataset']}",
    "checkpoint": f"/Volumes/{CONFIG['catalog']}/{bronze_schema}/{CONFIG['dataset']}/_checkpoints"
}

### 2. Idempotent Ingestion Pipeline (Auto Loader)
This function reads incoming files incrementally via cloudFiles and appends audit metadata safely.

In [0]:
def ingest_bronze_stream(source_path: str, checkpoint_path: str, target_table_name: str):
    print(f"Initializing Auto Loader stream from: {source_path}")
    
    # Configure incremental read stream
    source_stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema_evolution")
        .load(source_path)
    )
    
    # Enrich target dataset with required audit columns
    enriched_stream = (
        source_stream
        .withColumn("source_filename", col("_metadata.file_name"))
        .withColumn("ingested_at", current_timestamp())
        .withColumn("ingested_date", current_date())
    )
    
    # Write stream out to Delta table idempotently
    print(f"Writing stream to target table: {target_table_name}")
    active_query = (
        enriched_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{checkpoint_path}/stream_checkpoint")
        .trigger(availableNow=True)
        .toTable(target_table_name)
    )
    
    active_query.awaitTermination()
    print("Stream ingestion cycle completed successfully.")

# Execute pipeline
ingest_bronze_stream(
    source_path=PATHS["source_volume"],
    checkpoint_path=PATHS["checkpoint"],
    target_table_name=target_table
)

In [0]:
# Display a slice of the newly ingested target table
display(spark.table(target_table).limit(10))
print(f"Total record count in Bronze: {spark.table(target_table).count()}")

# [Job run link](https://adb-7405604503619901.1.azuredatabricks.net/jobs/435230727958493/runs/813123040703242?o=7405604503619901)